In [ ]:
## CSS FOR SLIDES (OPTIONAL)
try:
    from traitlets.config.manager import BaseJSONConfigManager
    import os
    path = os.path.expanduser("~/.jupyter/nbconfig")
    cm = BaseJSONConfigManager(config_dir=path)
    cm.update('livereveal', {
        'theme': 'Black',
        'transition': 'convex',
        'start_slideshow_at': 'selected',
        'scroll': True,
    })
except Exception as e:
    pass


In [ ]:
## PREAMBLE FOR CONFIGURATIONS
import warnings
warnings.filterwarnings("ignore") # prevent python from printing out ugly warnings for presentation
import numpy as np # import array package
import cv2 # import image manipulation package
import os, sys
base_path = os.getcwd() # RRtoolbox root path
sys.path.append(os.path.abspath(base_path)) # add RRtoolbox package path
sys.path.append(os.path.join(os.path.abspath(base_path),"thesis")) # add graphs' path

# using Controlstdout context (i.e with Controlstdout():) to prevent outputs to stdout
from RRtoolbox.lib.root import Controlstdout
#from RRtoolbox.lib.cache import MemoizedDict

# global configuration for images
from graphs.preamble import GraphTexData
from matplotlib.pyplot import * # import pylab for plots
setspath = os.path.join(base_path,"results/") # it is better to give absolute paths for caching
GraphTexData.saves = False # disable saving graphs
GraphTexData.shows = True # enable showing graphs

In [ ]:
from imrestore_steps import shell_steps

# Implementation of Algorithms for the Restoration of Retinal Images
    
## David Toro

1. Call __init__ -> Initialize varibales:
	- start profiler: Logs the time for each process
	- get list of images to restore
	- start pool of processes (if asked): Multitreading and Multiprocessing supporting pool to speedup processing
	- start feature class and configurations
	- initialize selection methods
	- get distanceThresh (default = 0.75): Used in the ratio test to filter out matches as suggested by D. Lowe
	- get inlineThresh (default = 0.2; range[0-1]): Threshold for testing number of inlines respect with total matches
	- get rectangularityThresh (default = 0.5; range[0-1]): Threshold for testing transformed corners of merging image $I_m$ transformed by homography
	- get ransacReprojThreshold (default = 5.0): Maximum allowed reprojection error to treat a point pair as an inlier for homography $\| \texttt{dstPoints} _i -  \texttt{convertPointsHomogeneous} ( \texttt{H} * \texttt{srcPoints} _i) \|  >  \texttt{ransacReprojThreshold}$ http://docs.opencv.org/2.4/modules/calib3d/doc/camera_calibration_and_3d_reconstruction.html#findhomography
	- get minKps (default = 4): minimun number of keypoints in image to process
	- get baseImage Instructions: Image from which the restored image
2. Call restore(self)
	- self.pre_selection()
		- Select baseImage based on keypoints (VARIANTS)
			- compute key-points running for each path self.compute_keypoint(path) 
				- check and get from cache or do:
					- Create mask using mod_retinal_mask which process using retinal mask if background (<20) pixels sum is less than the 10% of the image size or returns None
					- apply detectAndCompute(image, mask) : (Using ASIFT) (VARIANTS) http://docs.opencv.org/3.0-beta/doc/py_tutorials/py_feature2d/py_matcher/py_matcher.html
						- (ASIFT) create more affine transformations for the image $T_t$ making translations $t$ every $t = 2^{(0.5*n)}$ where n=[1-6] and rotations $\phi$ from 0 to 180 every $72/t$ steps. That is transform the image, its mask (if any) and keep the transformation matrix $TM$ to transform back to the original image $I$.
						- for each new affine transformation from ASIFT detect image keypoints and detect the features using any feature detector.
						- Transform keypoin positions ASIFT additional images using $TM$ to move these keypoints back to the domain of the original image $I$.
					- save into cache
				- Transform points using getSOpointRelation and cv2.perspectiveTransform
		- add baseImage to USED list
		- assign baseImage as restored image $I_r$ which will be updated later
	- self.matching()
		- Organize matching images based on most matched keypoints (VARIANTS)
			- Apply ratio test as D. Lowe suggestion in raw matches
			- make list of paths of ordered images from Most matches to least
	- Find Homography $H$ from ordered list and matches
	- Test Homography was found
	- Get transformed corners of merging image $I_m$
	- Test using inlineThresh for inline ratio and rectangularityThresh for rotatedRectangularity
	- Merge merging image $I_m$ into restored image $I_r$
		- Transform $I_m$ using $H$
		- Get foregraound mask using post_process_fore_Mask(self, back, fore):
			- Resize $back$ and $fore$ to small images to improve processing times
			- get $alpha_{mask}$ using :func:`get_layered_alpha`(back, fore)
				-
			- if GROW_SCENE adpat alpha mask as $alpha_{mask}[n<20] = max(alpha_{mask})$
			- Resize $alpha_{mask}$ back to the original sizes
		- Apply :func:`Overlay` as $back = fore*alpha_{mask} + back*(1-alpha_{mask})$ and assign to $I_r$
		- Update $fore$ and $back$ keypoints to be transformed in the space of $I_r$
			- get getTransformedCorners from $H$
			- use cv2.pointPolygonTest to know which are inside $fore$ to be transformed or $back$ to be left untouched
			- update self.kps_base and self.desc_base which contain $I_r$ keypoints and descriptors
		- add $I_m$ path to USED list
	- Check other images to merge in loop
3. Call post_process_restoration(self, image)
	- denoise $I_r$ using cv2.bilateralFilter with automatic parameters from :func:`getBilateralParameters`
	- if lens use :func:`simulateLens` function
		-
	- if enclose (or crop where object is)
        	- convert to gray using :func:`brightness`(image)
        	- get object mask using :func:`foreground`(gray)
            	- get contour of biggest area using :func:`thresh_biggestCnt`(mask)
            	- crop where biggest area is
4. Save image

In [ ]:
from graphs import graph_bilateralFilter, graph_bilateralFilter_parameters, \
    graph_convexityRatio, graph_numpy_image,graph_CLAHE, graph_decomposition, graph_entropy, \
    graph_histogram_matching, graph_overlay, graph_polygonArea, graph_rectangularity, graph_result, \
    graph_sift_descriptros, graph_smooth_hist_max_min, graph_spacial_filters, graph_thresholds, \
    graph_filter_prototypes, graph_find_brightest,graph_get_bright_alpha, graph_get_layered_alpha, \
    graph_histogram_equalization, graph_histogram_comparison, graph_features
from graphs.preamble import *
GraphTexData.graphs_buffer = [] # restart buffer if run several times
GraphTexData.saves = False
GraphTexData.shows = False
GraphTexData.buffered = True

In [ ]:
from graphs.graph_features import graph, GraphTexData
context = {"path":"/mnt/4E443F99443F82AF/Dropbox/PYTHON/RRtools/thesis/images",
           "fn":'patterns.png'}#,"ext":"svg"}
context["lookpath"] = "/mnt/4E443F99443F82AF/Dropbox/PYTHON/RRtools/tests"
data = graph(context,feature_name = "SIFT",fn="lena.png",useTitle=True,
             shapes=[(100,100,-30),(300,300),(300,300,30)],color=(150,0,0), onefig=True)

In [ ]:
from thesis.graphs.all import graph
context = {"path": os.path.join(base_path, "RRtoolbox", "lib", "arrayops", "filters.py")}


In [ ]:
len(GraphTexData.graphs_buffer)

In [ ]:
for buff in GraphTexData.graphs_buffer:
    buff["fig"].show()


In [ ]:
myFigure = Figure()
plot([3,2,1])

In [ ]:
%matplotlib notebook
myFigure

In [ ]:
from IPython.display import display
display(myFigure)

In [ ]:
fig, ax = subplots()
ax.plot([1,2,3,6],[4,2,3,4]);

In [ ]:
ax.plot([3.3, 4.4, 4.5, 6.5], [3., 5., 6., 7.])
fig 

In [ ]:
GraphTexData.graphs_buffer[0]["fig"]

In [ ]:
GraphTexData.graphs_buffer[1]["fig"]